# Person 4 — Join Pipeline (fast path)

Uses your shared team Drive folder:
`https://drive.google.com/drive/folders/1-Qr7yR49vehSs-Ddca3lUiZtidyid5q0`

**Before Run all (one-time):**
1. Open that Drive link while signed into the Google account you’ll use in Colab.
2. If the folder is under **Shared with me**, right-click it → **Organize → Add shortcut → My Drive**.
3. Upload this notebook to Colab → **Runtime → Run all** → approve Drive access.

Prefer **`Pilot 100 data`** for the 100-row test join. Falls back to searching the whole project folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import csv
from collections import defaultdict, Counter
from statistics import mean
from datetime import datetime, timezone

MYDRIVE = Path('/content/drive/MyDrive')
assert MYDRIVE.exists(), 'Drive mount failed'

# Folder id from your shared link
FOLDER_ID = '1-Qr7yR49vehSs-Ddca3lUiZtidyid5q0'

def find_project_root() -> Path:
    candidates = []
    # Common names / shortcut targets under MyDrive
    for name in ['MKT4220_Project', 'Pilot 100 data']:
        p = MYDRIVE / name
        if p.exists():
            candidates.append(p)
    # Any folder that contains Pilot 100 data or the known review file
    for p in MYDRIVE.rglob('Pilot 100 data'):
        if p.is_dir():
            candidates.append(p.parent if p.name == 'Pilot 100 data' else p)
            candidates.append(p)
    for p in MYDRIVE.rglob('reviews_image_filtered_v2.jsonl'):
        candidates.append(p.parents[min(2, len(p.parents)-1)])

    # Deduplicate
    uniq = []
    seen = set()
    for c in candidates:
        key = str(c.resolve()) if c.exists() else str(c)
        if key not in seen and Path(c).exists():
            seen.add(key)
            uniq.append(Path(c))

    if not uniq:
        raise FileNotFoundError(
            'Could not find the shared project under MyDrive.\n'
            'Open the team Drive folder → Organize → Add shortcut → My Drive, then re-run.\n'
            f'Folder: https://drive.google.com/drive/folders/{FOLDER_ID}'
        )

    # Prefer a parent that has both data/ and Pilot 100 data/
    for c in uniq:
        if (c / 'Pilot 100 data').exists() or (c / 'data').exists():
            print('Project root:', c)
            return c
    print('Project root:', uniq[0])
    return uniq[0]

PROJECT = find_project_root()
PILOT = PROJECT / 'Pilot 100 data'
if not PILOT.exists():
    # Maybe PROJECT itself is the Pilot folder
    if PROJECT.name == 'Pilot 100 data':
        PILOT = PROJECT
    else:
        hits = list(PROJECT.rglob('Pilot 100 data'))
        PILOT = hits[0] if hits else None

OUT = PROJECT / 'data' / 'joined_v2' if (PROJECT / 'data').exists() else PROJECT / 'joined_v2'
OUT.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = []
if PILOT and Path(PILOT).exists():
    SEARCH_ROOTS.append(Path(PILOT))
    print('Using Pilot 100 folder:', PILOT)
SEARCH_ROOTS.append(PROJECT)

def find_file(filename: str) -> Path:
    matches = []
    for root in SEARCH_ROOTS:
        matches.extend(sorted(root.rglob(filename)))
    # unique preserve order
    uniq = []
    seen = set()
    for m in matches:
        s = str(m)
        if s not in seen:
            seen.add(s)
            uniq.append(m)
    if not uniq:
        raise FileNotFoundError(
            f'Could not find {filename} under {SEARCH_ROOTS}. '
            'Check the Pilot 100 data folder contents.'
        )
    chosen = uniq[0]
    print(f'{filename} -> {chosen}')
    return chosen

# Try common filenames; text file may also be named slightly differently
REVIEWS_SRC = find_file('reviews_image_filtered_v2.jsonl')
try:
    TEXT_SRC = find_file('text_features_image_v2_test.jsonl')
except FileNotFoundError:
    # fallback patterns for Person-2 test exports
    alt = list(PROJECT.rglob('*text_features*v2*test*.jsonl')) + list(PROJECT.rglob('*text_features*.jsonl'))
    if not alt:
        raise
    TEXT_SRC = alt[0]
    print('text features (fallback) ->', TEXT_SRC)

IMAGES_SRC = find_file('image_features_available_v2.jsonl')
print('Output folder:', OUT)

## 1. Load text-feature IDs

In [ ]:
def load_jsonl(path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

text_rows = load_jsonl(TEXT_SRC)
target_ids = {r['review_id'] for r in text_rows}
print('text rows:', len(text_rows))
print('unique review_ids:', len(target_ids))
print('text run_ids:', sorted({r.get('run_id') for r in text_rows}))
assert len(text_rows) == len(target_ids), 'duplicate text review_ids'

## 2. Subset reviews + image features

In [ ]:
reviews = []
with REVIEWS_SRC.open('r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        r = json.loads(line)
        if r['review_id'] in target_ids:
            reviews.append(r)

images = []
with IMAGES_SRC.open('r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        r = json.loads(line)
        if r['review_id'] in target_ids:
            if 'image_source' not in r:
                r['image_source'] = 'customer_review'
            images.append(r)

review_ids = {r['review_id'] for r in reviews}
missing = sorted(target_ids - review_ids)
print('matched reviews:', len(reviews))
print('matched image rows:', len(images))
print('missing reviews for text ids:', len(missing))
if missing[:5]:
    print('missing examples:', missing[:5])
assert not missing, 'Every text review_id must exist in reviews file'

review_by_id = {r['review_id']: r for r in reviews}
reviews = [review_by_id[t['review_id']] for t in text_rows]

## 3. Join into 04_review_features

In [ ]:
text_by_id = {t['review_id']: t for t in text_rows}
images_by_review = defaultdict(list)
for img in images:
    images_by_review[img['review_id']].append(img)

batch_image_run_ids = {img.get('run_id') for img in images if img.get('run_id')}
batch_image_run_id = next(iter(batch_image_run_ids)) if len(batch_image_run_ids) == 1 else None

joined = []
status_counts = Counter()

for review in reviews:
    rid = review['review_id']
    text = text_by_id[rid]
    imgs = images_by_review.get(rid, [])

    ok_scores = [
        img['text_image_cosine']
        for img in imgs
        if img.get('status') == 'ok' and img.get('text_image_cosine') is not None
    ]
    usable = len(ok_scores)
    total = len(imgs) if imgs else int(review.get('image_count') or 0)

    if total == 0:
        image_status = 'no_images'
    elif usable == 0:
        image_status = 'failed'
    elif usable < total:
        image_status = 'partial'
    else:
        image_status = 'ok'
    status_counts[image_status] += 1

    run_ids = {img.get('run_id') for img in imgs if img.get('run_id')}
    if len(run_ids) == 1:
        image_run_id = next(iter(run_ids))
    elif imgs:
        image_run_id = sorted(imgs, key=lambda x: x.get('image_index', 0))[0].get('run_id')
    else:
        image_run_id = batch_image_run_id

    joined.append({
        'review_id': rid,
        'parent_asin': review['parent_asin'],
        'split': review['split'],
        'rating': review['rating'],
        'helpful_vote': review['helpful_vote'],
        'verified_purchase': review.get('verified_purchase'),
        'timestamp_ms': review['timestamp_ms'],
        'image_count': review['image_count'],
        'text_word_count': text.get('text_word_count'),
        'sentiment_negative': text.get('sentiment_negative'),
        'sentiment_neutral': text.get('sentiment_neutral'),
        'sentiment_positive': text.get('sentiment_positive'),
        'usable_image_count': usable,
        'image_status': image_status,
        'text_image_cosine_mean': mean(ok_scores) if ok_scores else None,
        'text_image_cosine_min': min(ok_scores) if ok_scores else None,
        'text_run_id': text.get('run_id'),
        'image_run_id': image_run_id,
    })

print('joined rows:', len(joined))
print('image_status:', dict(status_counts))
assert len(joined) == len(reviews) == len(text_rows)

## 4. Save outputs

In [ ]:
fields = list(joined[0].keys())
jsonl_path = OUT / '04_review_features.jsonl'
csv_path = OUT / '04_review_features.csv'
report_path = OUT / 'acceptance_report.json'

with jsonl_path.open('w', encoding='utf-8') as f:
    for row in joined:
        f.write(json.dumps(row) + '\n')

with csv_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for row in joined:
        out = {k: ('' if v is None else v) for k, v in row.items()}
        writer.writerow(out)

report = {
    'passed': True,
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'counts': {
        'reviews': len(reviews),
        'text_features': len(text_rows),
        'image_features': len(images),
        'joined_rows': len(joined),
        'image_status': dict(status_counts),
    },
    'sources': {
        'reviews': str(REVIEWS_SRC),
        'text': str(TEXT_SRC),
        'images': str(IMAGES_SRC),
    },
}
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')

print('DONE')
print('Saved:', jsonl_path)
print('Saved:', csv_path)
print('Saved:', report_path)
print('Preview:')
for row in joined[:3]:
    print({
        k: row[k]
        for k in [
            'review_id', 'split', 'image_status',
            'usable_image_count', 'text_image_cosine_mean', 'sentiment_positive'
        ]
    })